# Hurricane Tracker -- Interactive
This demonstrates the use of input and display widgets in a Jupyter Notebook

In [ ]:
# Import packages
from pathlib import Path
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# GIS pakcages
from arcgis import GIS
from arcgis.features import FeatureLayer
from arcgis.geometry import Polyline
from arcgis.geometry.filters import intersects

# Initialize ArcGIS GIS connection
gis = GIS('home')

# Disable URL warnings
import urllib3
urllib3.disable_warnings()

#Set paths
raw_folder_path = Path.cwd().parent / 'data' / 'raw'
processed_folder_path = Path.cwd().parent / 'data' / 'processed'

In [ ]:
#Set model layers
ibtracs_NA_points = raw_folder_path /'IBTrACS_NA.shp'
if not ibtracs_NA_points.exists():
    print('ibtracs_NA_points.shp file not found.\nBe sure you have unzipped the file.')
    
usa_counties = 'https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_Counties_Generalized_Boundaries/FeatureServer/0'

In [ ]:
#Convert storm point feature class to spatial dataframe
sdf_allstorms = pd.DataFrame.spatial.from_featureclass(ibtracs_NA_points)

In [ ]:
#Create a spatial dataframe from the counties
counties_lyr = FeatureLayer(usa_counties,gis=gis)

In [ ]:
#Function to extract affected counties for a given season and name
def get_affected_counties(the_season, the_name):
    #Select the points
    sdf_storm_points = sdf_allstorms.loc[
        (sdf_allstorms['SEASON']==the_season) &
        (sdf_allstorms['NAME']==the_name)
        ]

    #Sort points on ISO_TIME
    sdf_storm_points = sdf_storm_points.sort_values(by='ISO_TIME')

    #Create a storm track line from the points
    coords = [[geom.x, geom.y]  for geom in sdf_storm_points["SHAPE"]]
    track_line = Polyline({
        'paths':[coords],
        'spatialReference':sdf_storm_points.spatial.sr
    })

    #Query the counties feature layer for records intersecting the track line geometry
    intersecting_counties = counties_lyr.query(
        geometry_filter=intersects(track_line)
    ).sdf

    #Abort if no intersecting counties are found
    if intersecting_counties.empty:
        return None
    
    #Plot the population affected
    the_plot = intersecting_counties.groupby('STATE_NAME')['POPULATION'].sum().sort_values().plot.barh();

    return the_plot


### Create the widgets

In [ ]:
#Set initial values
the_season = 2005
the_name = 'KATRINA'

In [ ]:
#Create a dropdown widget for selecting a storm season
season_dropdown = widgets.Dropdown(
    options=sdf_allstorms['SEASON'].unique(),
    value=the_season,
    description='Season:',
)

In [ ]:
#Create a dropdown widget for selecting a storm name
name_dropdown = widgets.Dropdown(
    options=sdf_allstorms[sdf_allstorms['SEASON'] == the_season]['NAME'].unique(),
    value=the_name,
    description='Name:',
)

In [ ]:
#Add reaction to the season_dropdown so that when it changes, the name_dropdown options update accordingly
def update_name_dropdown(change):
    season = change['new']
    name_dropdown.options = sdf_allstorms[sdf_allstorms['SEASON'] == season]['NAME'].unique()

season_dropdown.observe(update_name_dropdown, names='value')

### Add reactivity

In [ ]:
#Create a space to show the plot generated
plot_output = widgets.Output()

#Function to update the plot, called when widget values change
def update_data_or_plot(change=None):
    #Direct output to the plot_output widget
    with plot_output:
        # Clear the previous plot
        clear_output(wait=False)

        # Generate the new plot
        g = get_affected_counties(
            season_dropdown.value,
            name_dropdown.value
        )

        #Abort if no intersecting counties are found
        if g is None:
            print("No intersecting counties found.")
            return

        # Explicitly display the plot
        display(g.figure)
        plt.close(g.figure)

#Attach observers   
name_dropdown.observe(update_data_or_plot, names='value')
season_dropdown.observe(update_data_or_plot, names='value')

#Display the widgets and plot together
display(
    widgets.HBox([season_dropdown, name_dropdown]),
    plot_output
)

#Generate the initial plot
update_data_or_plot()